<a href="https://colab.research.google.com/github/Stanley-Lam/DATA-4620-HW1-Tree-Species-Classification/blob/main/hw1_tree_species_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import os

In [ ]:
if not os.path.exists('tree_species_classifier_data.npz'):
  !wget -O tree_species_classifier_data.npz "https://www.dropbox.com/scl/fi/b7mw23k3ifaeui9m8nnn3/tree_species_classifier_data.npz?rlkey=bgxp37c1t04i7q35waf3slc26&dl=1"

In [ ]:
data = np.load('tree_species_classifier_data.npz')
train_features = data['train_features']
train_labels = data['train_labels']
test_features = data['test_features']
test_labels = data['test_labels']

##Exploratory

In [ ]:
print(data.files)

['train_features', 'train_labels', 'test_features', 'test_labels']


In [ ]:
print(train_features.shape)
print(train_labels.shape)
print(test_features.shape)
print(test_labels.shape)

(15707, 426)
(15707,)
(1554, 426)
(1554,)


In [ ]:
print(train_features.dtype)
print(train_labels.dtype)
print(test_features.dtype)
print(test_labels.dtype)

int16
uint8
int16
uint8


Rows are tree examples and columns are hyperspectral features/bands

In [ ]:
print("X_train min:", train_features.min())
print("X_train max:", train_features.max())
print("X_train range:", train_features.max() - train_features.min())

print("X_test min:", test_features.min())
print("X_test max:", test_features.max())
print("X_test range:", test_features.max() - test_features.min())

print("y_train min:", train_labels.min())
print("y_train max:", train_labels.max())

print("y_test min:", test_labels.min())
print("y_test max:", test_labels.max())

X_train min: 0
X_train max: 14998
X_train range: 14998
X_test min: 0
X_test max: 6908
X_test range: 6908
y_train min: 0
y_train max: 7
y_test min: 0
y_test max: 7


There are 8 classes or 8 different forest trees species

In [ ]:
classes, train_counts = np.unique(train_labels, return_counts = True)

print("Training set:")
for cls, count in zip(classes, train_counts):
    print(cls, count)

Training set:
0 2519
1 821
2 1575
3 3980
4 2640
5 88
6 852
7 3232


In [ ]:
classes, test_counts = np.unique(test_labels, return_counts = True)

print("Test set:")
for cls, count in zip(classes, test_counts):
    print(cls, count)

Test set:
0 389
1 30
2 278
3 404
4 100
5 22
6 43
7 288


##Preprocess Data via PCA

In [ ]:
pca = PCA(n_components = 32, whiten = True)

train_features_pca = pca.fit_transform(train_features)
test_features_pca = pca.transform(test_features)

print("Original training shape:", train_features.shape)
print("PCA training shape:", train_features_pca.shape)
print(" ")
print("Original test shape:", test_features.shape)
print("PCA test shape:", test_features_pca.shape)

Original training shape: (15707, 426)
PCA training shape: (15707, 32)
 
Original test shape: (1554, 426)
PCA test shape: (1554, 32)


In [ ]:
label_encoder = LabelEncoder()

train_labels_encoded = label_encoder.fit_transform(train_labels)
test_labels_encoded = label_encoder.transform(test_labels)

print("Classes:", label_encoder.classes_)
print("Encoded classes:", np.unique(train_labels_encoded))

Classes: [0 1 2 3 4 5 6 7]
Encoded classes: [0 1 2 3 4 5 6 7]


##Classifiers using scikit-learn

In [ ]:
linear_clf = LogisticRegression(max_iter = 1000)

linear_clf.fit(train_features_pca, train_labels_encoded)

linear_predictions = linear_clf.predict(test_features_pca)

linear_accuracy = accuracy_score(
    test_labels_encoded,
    linear_predictions
)

print("Scikit-learn Linear Test Accuracy:", round(linear_accuracy, 5))

Scikit-learn Linear Test Accuracy: 0.83398


In [ ]:
nn_clf = MLPClassifier(
    hidden_layer_sizes = (100,),
    max_iter = 1000,
    random_state = 42
)

nn_clf.fit(train_features_pca, train_labels_encoded)

nn_predictions = nn_clf.predict(test_features_pca)

nn_accuracy = accuracy_score(
    test_labels_encoded,
    nn_predictions
)

print("Scikit-learn NN Test Accuracy:", round(nn_accuracy, 5))

Scikit-learn NN Test Accuracy: 0.80309


##Classifiers using PyTorch

In [ ]:
train_features_tensor = torch.tensor(
    train_features_pca,
    dtype = torch.float32
)

test_features_tensor = torch.tensor(
    test_features_pca,
    dtype = torch.float32
)

train_labels_tensor = torch.tensor(
    train_labels_encoded,
    dtype = torch.long
)

test_labels_tensor = torch.tensor(
    test_labels_encoded,
    dtype = torch.long
)

In [ ]:
train_dataset = TensorDataset(
    train_features_tensor,
    train_labels_tensor
)

test_dataset = TensorDataset(
    test_features_tensor,
    test_labels_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle = True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle = False
)

In [ ]:
def calculate_accuracy(model, data_loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in data_loader:
            outputs = model(X_batch)
            predictions = torch.argmax(outputs, dim = 1)
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    accuracy = correct / total

    return accuracy

In [ ]:
def train_model(model, train_loader, test_loader):

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr = 1e-2,
        weight_decay = 0.001
    )

    for epoch in range(100):

        model.train()

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

        train_accuracy = calculate_accuracy(
            model,
            train_loader
        )
        test_accuracy = calculate_accuracy(
            model,
            test_loader
        )

        print(
            f"Epoch {epoch + 1} | "
            f"Train Accuracy: {train_accuracy:.4f} | "
            f"Test Accuracy: {test_accuracy:.4f}"
        )

    return model

In [ ]:
num_features = train_features_pca.shape[1]
num_classes = len(np.unique(train_labels_encoded))

linear_model = nn.Linear(
    num_features,
    num_classes
)

linear_model = train_model(
    linear_model,
    train_loader,
    test_loader
)

Epoch 1 | Train Accuracy: 0.7529 | Test Accuracy: 0.7400
Epoch 2 | Train Accuracy: 0.7882 | Test Accuracy: 0.7838
Epoch 3 | Train Accuracy: 0.8059 | Test Accuracy: 0.7967
Epoch 4 | Train Accuracy: 0.8147 | Test Accuracy: 0.8037
Epoch 5 | Train Accuracy: 0.8212 | Test Accuracy: 0.8044
Epoch 6 | Train Accuracy: 0.8261 | Test Accuracy: 0.8076
Epoch 7 | Train Accuracy: 0.8295 | Test Accuracy: 0.8082
Epoch 8 | Train Accuracy: 0.8313 | Test Accuracy: 0.8102
Epoch 9 | Train Accuracy: 0.8329 | Test Accuracy: 0.8108
Epoch 10 | Train Accuracy: 0.8344 | Test Accuracy: 0.8153
Epoch 11 | Train Accuracy: 0.8347 | Test Accuracy: 0.8160
Epoch 12 | Train Accuracy: 0.8354 | Test Accuracy: 0.8172
Epoch 13 | Train Accuracy: 0.8364 | Test Accuracy: 0.8172
Epoch 14 | Train Accuracy: 0.8375 | Test Accuracy: 0.8172
Epoch 15 | Train Accuracy: 0.8384 | Test Accuracy: 0.8198
Epoch 16 | Train Accuracy: 0.8382 | Test Accuracy: 0.8198
Epoch 17 | Train Accuracy: 0.8393 | Test Accuracy: 0.8192
Epoch 18 | Train Accura

In [ ]:
linear_train_accuracy = calculate_accuracy(
    linear_model,
    train_loader
)

linear_test_accuracy = calculate_accuracy(
    linear_model,
    test_loader
)

print("Final Linear Train Accuracy:", round(linear_train_accuracy, 4))
print("Final Linear Test Accuracy:", round(linear_test_accuracy, 4))

Final Linear Train Accuracy: 0.8483
Final Linear Test Accuracy: 0.8269


In [ ]:
class NeuralNetwork(nn.Module):

    def __init__(self, num_features, num_classes):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(num_features, 100),
            nn.ReLU(),
            nn.Linear(100, num_classes)
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
nn_model = NeuralNetwork(
    num_features,
    num_classes
)

nn_model = train_model(
    nn_model,
    train_loader,
    test_loader
)

Epoch 1 | Train Accuracy: 0.6621 | Test Accuracy: 0.6512
Epoch 2 | Train Accuracy: 0.7641 | Test Accuracy: 0.7812
Epoch 3 | Train Accuracy: 0.8021 | Test Accuracy: 0.8063
Epoch 4 | Train Accuracy: 0.8187 | Test Accuracy: 0.8121
Epoch 5 | Train Accuracy: 0.8320 | Test Accuracy: 0.8147
Epoch 6 | Train Accuracy: 0.8406 | Test Accuracy: 0.8218
Epoch 7 | Train Accuracy: 0.8461 | Test Accuracy: 0.8275
Epoch 8 | Train Accuracy: 0.8503 | Test Accuracy: 0.8308
Epoch 9 | Train Accuracy: 0.8540 | Test Accuracy: 0.8327
Epoch 10 | Train Accuracy: 0.8579 | Test Accuracy: 0.8314
Epoch 11 | Train Accuracy: 0.8607 | Test Accuracy: 0.8385
Epoch 12 | Train Accuracy: 0.8608 | Test Accuracy: 0.8391
Epoch 13 | Train Accuracy: 0.8631 | Test Accuracy: 0.8423
Epoch 14 | Train Accuracy: 0.8643 | Test Accuracy: 0.8449
Epoch 15 | Train Accuracy: 0.8655 | Test Accuracy: 0.8436
Epoch 16 | Train Accuracy: 0.8674 | Test Accuracy: 0.8456
Epoch 17 | Train Accuracy: 0.8683 | Test Accuracy: 0.8468
Epoch 18 | Train Accura

In [ ]:
nn_train_accuracy = calculate_accuracy(
    nn_model,
    train_loader
)

nn_test_accuracy = calculate_accuracy(
    nn_model,
    test_loader
)

print("Final NN Train Accuracy:", round(nn_train_accuracy, 4))
print("Final NN Test Accuracy:", round(nn_test_accuracy, 4))

Final NN Train Accuracy: 0.9204
Final NN Test Accuracy: 0.8616


In [ ]:
print("MODEL RESULTS")

print("Scikit-learn Linear:", round(linear_accuracy, 5))
print("Scikit-learn NN:", round(nn_accuracy, 5))
print("PyTorch Linear:", round(linear_test_accuracy, 5))
print("PyTorch NN:", round(nn_test_accuracy, 5))

MODEL RESULTS
Scikit-learn Linear: 0.83398
Scikit-learn NN: 0.80309
PyTorch Linear: 0.8269
PyTorch NN: 0.86165
